# Day 03 · Filtering, expressions and retail amounts

**Priority version:** 10 tasks that cover every unique concept of the day (all 5 assignments, 5 extra examples and 30 questions are mapped below). The last cell answers the conceptual questions.

**Course definitions (DATA_DICTIONARY.md):**
- `line_gross = quantity × unit_price`
- `line_net = round(line_gross × (1 − discount_pct), 2)`, rounding **half-up**

| Task | Concept | Covers |
|---|---|---|
| 1 | Type the columns, then compute gross and net | A1, A2, Q3, Q4, Q10, Q11, Q12, Q20 |
| 2 | Filter valid lines with `&`, `\|` and parentheses | A3, Q5, Q6, Q7, Q14, Q18 |
| 3 | Classify amounts with `when` / `otherwise` | A4, Q8, Q15, Q26 |
| 4 | Nulls: `isNull()` vs `== None`, and null inputs | A5, Q9, Q19, Q25 |
| 5 | Rounding: half-up vs `bround`, per line vs on the total | Q13, Q21, Q22 |
| 6 | Decimal precision and overflow (ANSI on/off) | Q23, Q24 |
| 7 | `select` vs `withColumn`, and how Spark simplifies the plan | Q1, Q2, Q17, Q29 |
| 8 | Built-in expressions vs a Python loop | Q28 |
| 9 | A calculation function with boundary checks | Q30, E2, E5 |
| 10 | Quick extras: > 2000, negative quantity, change an assumption | E1, E3, E4, Q16, Q27 |

Run on the **pyspark-course** cluster, not Serverless.

In [ ]:
from pyspark.sql import functions as F
from decimal import Decimal, ROUND_HALF_UP

RAW = "/Volumes/dbw_pyspark_course/retail/raw"
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Schema for the small test DataFrames (fixtures) used in today's tasks
LINE_SCHEMA = "line_id STRING, quantity INT, unit_price DECIMAL(12,2), discount_pct DECIMAL(5,2)"

### Task 1 · Type the columns, then compute gross and net
**Covers:** A1, A2, Q3, Q4, Q10, Q11, Q12, Q20

**Problem:**
1. Show why CSV text must be typed before doing maths or comparisons on it.
2. Read `order_items.csv` as text, then `cast` the quantity to `INT` and the money columns to `DECIMAL`.
3. Write `line_gross` and `line_net` **once**, as named reusable expressions, and apply them to a 2 × 500 line with a 10% discount and to all the real lines.

**Expected:**
1. As text, `"10" > "5"` is **false**: text is compared character by character, and `"1"` comes before `"5"`. After casting to `INT`, 10 > 5 is true.
2. `cast` changes the column's **type**, not the file. Text that can't be converted becomes `null`.
3. The 2 × 500 line at 10% gives `line_gross = 1000.00` and **`line_net = 900.00`** (the completion check). `line_net` has type `decimal(12,2)`.

In [ ]:
# 1. Why type before comparing (Q4): text is compared character by character
q = spark.createDataFrame([("9",), ("10",)], "quantity STRING")
q.filter(F.col("quantity") > "5").show()                 # text: only "9"
q.filter(F.col("quantity").cast("int") > 5).show()       # number: 9 and 10

# 2. Read as text, then cast (the starter pattern)
raw = spark.read.option("header", True).csv(f"{RAW}/order_items.csv")       # every column is a string
lines = (raw
         .withColumn("quantity", F.col("quantity").cast("int"))
         .withColumn("unit_price", F.col("unit_price").cast("decimal(12,2)"))
         .withColumn("discount_pct", F.col("discount_pct").cast("decimal(5,2)")))
lines.printSchema()

# 3. Named, reusable expressions (Q20): a Column object is just a formula until a DataFrame uses it
LINE_GROSS = F.col("quantity") * F.col("unit_price")
LINE_NET = F.round(LINE_GROSS * (F.lit(1) - F.col("discount_pct")), 2).cast("decimal(12,2)")   # F.round = half-up

example = spark.createDataFrame([("X1", 2, Decimal("500.00"), Decimal("0.10"))], LINE_SCHEMA)
example.select("line_id", LINE_GROSS.alias("line_gross"), LINE_NET.alias("line_net")).show()

amounts = lines.withColumn("line_gross", LINE_GROSS).withColumn("line_net", LINE_NET)
amounts.show(5)

### Task 2 · Filter valid lines
**Covers:** A3, Q5, Q6, Q7, Q14, Q18

**Problem:** Keep only lines with a **positive quantity**, a **non-negative price** and a `discount_pct` **between 0 and 1 inclusive**. Test it on a fixture with discounts 0, 1, −0.1 and 1.1 and quantities 0 and −1, then on the real data.

**Expected:**
- Fixture: only `D0` (net 1000.00) and `D1` (net 0.00, a 100% discount is still valid) survive.
- Without parentheses, `F.col("quantity") > 0 & ...` fails. Python evaluates `&` **before** `>`, so it tries `0 & <condition>`, and Spark rejects the plan with a `DATATYPE_MISMATCH` error.
- `filter` and `where` are the same function.
- Real data: **779** of 785 lines are valid. The 6 excluded are `BAD00`–`BAD02` (quantity −1) and `BAD03`–`BAD05` (discount 1.50).

In [ ]:
# Validity rule as a named expression. coalesce(..., False) turns "unknown" (null) into False for a clean flag column.
IS_VALID = F.coalesce(
    (F.col("quantity") > 0) & (F.col("unit_price") >= 0) & F.col("discount_pct").between(0, 1),
    F.lit(False))

disc_fx = spark.createDataFrame([
    ("D0",   2, Decimal("500.00"), Decimal("0.00")),
    ("D1",   2, Decimal("500.00"), Decimal("1.00")),
    ("DNEG", 2, Decimal("500.00"), Decimal("-0.10")),
    ("D11",  2, Decimal("500.00"), Decimal("1.10")),
    ("Q0",   0, Decimal("500.00"), Decimal("0.10")),
    ("QNEG", -1, Decimal("500.00"), Decimal("0.10")),
], LINE_SCHEMA)

disc_fx.where(IS_VALID).select("line_id", LINE_NET.alias("line_net")).show()   # where == filter

# Q7 · why parentheses matter: & binds tighter than > in Python
try:
    disc_fx.filter(F.col("quantity") > 0 & F.col("discount_pct").between(0, 1))
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0][:160])

# Real data
valid = lines.filter(IS_VALID)
print("valid lines:", valid.count(), "of", lines.count())
lines.filter(~IS_VALID).select("line_id", "quantity", "discount_pct").show()

### Task 3 · Classify amounts with `when` / `otherwise`
**Covers:** A4, Q8, Q15, Q26

**Problem:** Band each line by `line_net`: below 500 → `low`; 500 to 1999.99 → `medium`; 2000 and above → `high`. Test the exact boundaries 499.99, 500.00, 1999.99 and 2000.00, plus a `null`.

**Expected:**
- Boundaries: 499.99 → `low` · 500.00 → `medium` · 1999.99 → `medium` · 2000.00 → `high`.
- **Null trap:** in the naive version, `null < 500` is null rather than true, so a null amount falls all the way through to `otherwise` and is labelled **`high`**. The safe version checks for null first and gives `unknown`.
- Real valid lines: **low 56 · medium 329 · high 394**.

In [ ]:
def band(net):
    """Band a line_net column. The null check comes first, otherwise null would fall through to 'high'."""
    return (F.when(net.isNull(), "unknown")
             .when(net < 500, "low")
             .when(net < 2000, "medium")
             .otherwise("high"))

naive_band = F.when(F.col("line_net") < 500, "low").when(F.col("line_net") < 2000, "medium").otherwise("high")

boundaries = spark.createDataFrame(
    [(Decimal("499.99"),), (Decimal("500.00"),), (Decimal("1999.99"),), (Decimal("2000.00"),), (None,)],
    "line_net DECIMAL(12,2)")
boundaries.select("line_net", band(F.col("line_net")).alias("band"), naive_band.alias("naive_band")).show()

# Real data
valid.select(band(LINE_NET).alias("band")).groupBy("band").count().orderBy("band").show()

### Task 4 · Nulls
**Covers:** A5, Q9, Q19, Q25

**Problem:** On a fixture with a null quantity (`N2`), a null price (`N3`) and a null discount (`N4`):
1. Compare `isNull()` with `== None`.
2. Count the rows where `quantity > 0`, and the rows where it is **not** `> 0`.
3. Compute `line_net`. Why must you **not** fill missing values with 0?

**Expected:**
1. `== None` returns **0 rows**, because comparing anything with null gives null (unknown), never true. `isNull()` correctly finds `N2`. The null-safe equality `eqNullSafe(None)` also works.
2. `quantity > 0` → 3 rows, and `NOT (quantity > 0)` → **0 rows**. `N2` disappears from **both** sides: a null fails every comparison.
3. `line_net` is `null` for `N2`, `N3` and `N4`: null spreads through arithmetic. Filling the discount with 0 (`coalesce`) would charge `N4` the **full price**, 1000.00, while its real discount is unknown. Flag the line instead.

In [ ]:
null_fx = spark.createDataFrame([
    ("N1", 2,    Decimal("500.00"), Decimal("0.10")),
    ("N2", None, Decimal("500.00"), Decimal("0.10")),
    ("N3", 2,    None,              Decimal("0.10")),
    ("N4", 2,    Decimal("500.00"), None),
], LINE_SCHEMA)

# 1. isNull vs == None
print("== None        :", null_fx.filter(F.col("quantity") == None).count())          # noqa: E711 (on purpose)
print("isNull()       :", null_fx.filter(F.col("quantity").isNull()).count())
print("eqNullSafe     :", null_fx.filter(F.col("quantity").eqNullSafe(None)).count())

# 2. Nulls vanish from both sides of a comparison
print("quantity > 0   :", null_fx.filter(F.col("quantity") > 0).count())
print("NOT quantity>0 :", null_fx.filter(~(F.col("quantity") > 0)).count())

# 3. Handle missing inputs explicitly instead of pretending they are 0
missing_input = F.col("quantity").isNull() | F.col("unit_price").isNull() | F.col("discount_pct").isNull()
filled_net = F.round(LINE_GROSS * (F.lit(1) - F.coalesce("discount_pct", F.lit(0))), 2)      # the wrong way
null_fx.select("line_id",
               LINE_NET.alias("line_net"),
               F.when(missing_input, "missing_input").otherwise("ok").alias("input_status"),
               filled_net.alias("wrong_if_filled_with_0")).show()

### Task 5 · Rounding
**Covers:** Q13, Q21, Q22

**Problem:**
1. Compare `round` (half-up, the course rule) with `bround` (half-even, "banker's rounding") on ties such as 2.345, 2.355 and 0.125.
2. Three lines each have an unrounded net of 0.335. Compare **rounding each line, then summing** with **summing first, then rounding the total**.

**Expected:**
1. `round`: 2.35 · 2.36 · 0.13. `bround`: **2.34** · 2.36 · **0.12**. On an exact tie (…5), `bround` rounds to the **even** neighbour, while `round` always rounds up.
2. Per line: 0.34 × 3 = **1.02**. Total first: round(1.005) = **1.01**. They differ by one paisa. The course rounds **each line** (DATA_DICTIONARY), so totals always equal the sum of the visible lines.

In [ ]:
# 1. Half-up vs half-even. SQL literals like 2.345 are DECIMAL, so there is no float noise.
spark.sql("""
    SELECT round(2.345, 2) AS round_2_345, bround(2.345, 2) AS bround_2_345,
           round(2.355, 2) AS round_2_355, bround(2.355, 2) AS bround_2_355,
           round(0.125, 2) AS round_0_125, bround(0.125, 2) AS bround_0_125
""").show()

# 2. Round per line vs round the total
tie_lines = spark.createDataFrame(
    [(f"T{i}", 1, Decimal("0.67"), Decimal("0.50")) for i in (1, 2, 3)], LINE_SCHEMA)   # 0.67 * 0.5 = 0.335
unrounded = LINE_GROSS * (F.lit(1) - F.col("discount_pct"))
tie_lines.agg(
    F.sum(LINE_NET).alias("round_each_line_then_sum"),
    F.round(F.sum(unrounded), 2).alias("sum_then_round_total"),
).show()

### Task 6 · Decimal precision and overflow
**Covers:** Q23, Q24

**Problem:**
1. Print the types of `line_gross`, of the **unrounded** net, and of the final `line_net`.
2. A line of 2,000,000,000 × 9,999,999.99 doesn't fit `DECIMAL(12,2)`. What happens with ANSI mode **off** and **on**?

**Expected:**
1. Precision grows with each step. Multiplying adds the precisions (+1) and the scales: `INT × DECIMAL(12,2)` → **`decimal(23,2)`**, and × `(1 − discount)` → **`decimal(30,4)`**. Our final `cast` brings it back to **`decimal(12,2)`**. If your exact numbers differ slightly, it's the Spark version; the point is that precision grows.
2. ANSI **off**: the result silently becomes **`null`**, which is dangerous because revenue quietly disappears. ANSI **on**: an error, `NUMERIC_VALUE_OUT_OF_RANGE` or `CAST_OVERFLOW`. A loud failure is safer for money.

In [ ]:
# 1. Intermediate decimal types
example.select(
    LINE_GROSS.alias("line_gross"),
    (LINE_GROSS * (F.lit(1) - F.col("discount_pct"))).alias("net_unrounded"),
    LINE_NET.alias("line_net"),
).printSchema()

# 2. Overflow with ANSI off and on
big = spark.createDataFrame([("BIG", 2000000000, Decimal("9999999.99"), Decimal("0.00"))], LINE_SCHEMA)
original_ansi = spark.conf.get("spark.sql.ansi.enabled")
for ansi in ["false", "true"]:
    spark.conf.set("spark.sql.ansi.enabled", ansi)
    try:
        print(f"ANSI {ansi:5} ->", big.select(LINE_NET.alias("line_net")).first()["line_net"])
    except Exception as e:
        print(f"ANSI {ansi:5} -> error:", str(e).splitlines()[0][:150])
spark.conf.set("spark.sql.ansi.enabled", original_ansi)

### Task 7 · `select` vs `withColumn`, and plan simplification
**Covers:** Q1, Q2, Q17, Q29

**Problem:**
1. Return only `line_id` and the computed amounts, leaving the raw columns untouched.
2. Chain three `withColumn` calls and look at the **optimized** plan. Does Spark run three separate steps?

**Expected:**
1. `select` returns **only** the columns you list (a new DataFrame), while `withColumn` keeps **all** columns and adds or replaces one. `lines` itself never changes.
2. No. In `== Optimized Logical Plan ==` the three steps are merged into **one `Project`**, with `a` and `b` substituted into `c`'s formula. The optimizer (Catalyst) collapses chained projections.

In [ ]:
# 1. select = choose exactly these columns
only_amounts = lines.select("line_id", LINE_GROSS.alias("line_gross"), LINE_NET.alias("line_net"))
only_amounts.show(3)
print("lines still has:", lines.columns)

# 2. Chained projections get collapsed
chained = (lines.withColumn("a", F.col("quantity") + 1)
                .withColumn("b", F.col("a") * 2)
                .withColumn("c", F.col("b") - 1))
chained.explain(True)

### Task 8 · Built-in expressions vs a Python loop
**Covers:** Q28

**Problem:** Compute the total net of the valid lines twice: once with Spark built-ins (`F.sum`), and once by `collect()`-ing the rows and looping in Python.

**Expected:** Both give **2,025,136.25**. The timing is meaningless on 779 rows. The real difference is **where** the work happens. Built-ins run in parallel on the executors and send back one number. The loop drags **every row** to the driver and adds them one at a time on a single core, which crashes or crawls on big data.

(This is not yet the course's *booked revenue*, 1,829,120.00. That figure also removes duplicates, unknown references and cancelled orders, which come on Days 04–06.)

In [ ]:
import time

t0 = time.time()
spark_total = valid.agg(F.sum(LINE_NET)).first()[0]
t_spark = time.time() - t0

t0 = time.time()
rows = valid.select("quantity", "unit_price", "discount_pct").collect()      # every row -> driver
python_total = sum((r.quantity * r.unit_price * (1 - r.discount_pct)).quantize(Decimal("0.01"), ROUND_HALF_UP)
                   for r in rows)
t_python = time.time() - t0

print(f"Spark built-ins: {spark_total}  ({t_spark:.2f}s)")
print(f"Python loop    : {python_total}  ({t_python:.2f}s, {len(rows)} rows pulled to the driver)")

### Task 9 · A calculation function with boundary checks
**Covers:** Q30, E2, E5

**Problem:** Wrap today's logic in `add_line_amounts(df)`, which adds `line_gross`, `line_net`, `is_valid` and `net_band`. Check it against a table of boundary cases with **expected** values. Then deliberately break it (use `bround` instead of the course's half-up rule) and show that the check catches it.

**Expected:**
- Correct function: **0 failed cases**. This includes the 2 × 500 at 10% line (E2) = 900.00.
- Broken function: **1 failed case**, `B6`. Its unrounded net is 0.325: half-up gives 0.33, half-even gives 0.32.

In [ ]:
def add_line_amounts(df):
    """Input: typed quantity, unit_price, discount_pct. Adds line_gross, line_net, is_valid, net_band."""
    return (df.withColumn("line_gross", LINE_GROSS)
              .withColumn("line_net", LINE_NET)
              .withColumn("is_valid", IS_VALID)
              .withColumn("net_band", band(F.col("line_net"))))

D = Decimal
cases = spark.createDataFrame([
    # line_id, qty, price,        discount,   exp_net,          exp_band,  exp_valid
    ("B1",  1,  D("499.99"),  D("0.00"),  D("499.99"),   "low",     True),
    ("B2",  1,  D("500.00"),  D("0.00"),  D("500.00"),   "medium",  True),
    ("B3",  1,  D("1999.99"), D("0.00"),  D("1999.99"),  "medium",  True),
    ("B4",  1,  D("2000.00"), D("0.00"),  D("2000.00"),  "high",    True),
    ("B5",  2,  D("500.00"),  D("0.10"),  D("900.00"),   "medium",  True),    # E2 / completion check
    ("B6",  1,  D("0.65"),    D("0.50"),  D("0.33"),     "low",     True),    # 0.325 -> half-up 0.33
    ("B7",  2,  D("500.00"),  D("1.10"),  D("-100.00"),  "low",     False),   # invalid discount
    ("B8",  -1, D("500.00"),  D("0.00"),  D("-500.00"),  "low",     False),   # negative quantity
    ("B9",  2,  None,         D("0.10"),  None,          "unknown", False),   # missing price
], LINE_SCHEMA + ", exp_net DECIMAL(12,2), exp_band STRING, exp_valid BOOLEAN")

def failed_cases(fn):
    out = fn(cases)
    return out.filter(~F.col("line_net").eqNullSafe(F.col("exp_net"))
                      | ~F.col("net_band").eqNullSafe(F.col("exp_band"))
                      | ~F.col("is_valid").eqNullSafe(F.col("exp_valid")))

add_line_amounts(cases).select("line_id", "line_net", "exp_net", "net_band", "exp_band", "is_valid", "exp_valid").show()
print("correct function, failed cases:", failed_cases(add_line_amounts).count())

def add_line_amounts_broken(df):        # half-even instead of the course's half-up rule
    broken_net = F.bround(LINE_GROSS * (F.lit(1) - F.col("discount_pct")), 2).cast("decimal(12,2)")
    return add_line_amounts(df).withColumn("line_net", broken_net)

failed_cases(add_line_amounts_broken).select("line_id", "line_net", "exp_net").show()

### Task 10 · Quick extras
**Covers:** E1, E3, E4, Q16, Q27

| | Problem | Expected |
|---|---|---|
| a | Valid lines with net **above 2000** (E1, Q16) | **385** with `> 2000`, but **394** with `>= 2000`: 9 lines are exactly 2000.00. Read the boundary wording carefully |
| b | How do **negative quantities** affect revenue? (E3, Q27) | Keeping them gives 2,022,511.25 instead of 2,025,136.25, so revenue is **understated by 2,625.00**. Returns live in the `returns` table; a negative quantity here is bad data, not a refund, and counting it would subtract money that was never refunded, or subtract a refund twice |
| c | **Change an assumption (E4):** instead of excluding lines with an invalid discount, treat their discount as 0 | Revenue rises by **10,500.00** to 2,035,636.25: `BAD03`–`BAD05` are counted at full price (3,500 + 875 + 6,125). One rule change moves the total |

In [ ]:
# (a) E1 / Q16 · above 2000: watch the boundary
print("net >  2000:", valid.filter(LINE_NET > 2000).count())
print("net >= 2000:", valid.filter(LINE_NET >= 2000).count())

# (b) E3 / Q27 · what negative quantities do to revenue
discount_ok = F.col("discount_pct").between(0, 1)
with_negatives = lines.filter(discount_ok).agg(F.sum(LINE_NET)).first()[0]
valid_total = valid.agg(F.sum(LINE_NET)).first()[0]
print("total incl. negative quantities:", with_negatives)
print("total of valid lines           :", valid_total, "| difference:", valid_total - with_negatives)

# (c) E4 · change an assumption: invalid discount -> treat as 0 instead of excluding the line
discount_or_zero = F.when(discount_ok, F.col("discount_pct")).otherwise(F.lit(0))
alt_net = F.round(LINE_GROSS * (F.lit(1) - discount_or_zero), 2)
alt_total = lines.filter(F.col("quantity") > 0).agg(F.sum(alt_net)).first()[0]
print("total if invalid discounts = 0 :", alt_total, "| change:", alt_total - valid_total)

### Concept notes (Q1–Q10, Q24, Q27): read once, then rewrite in your own words
- **select vs withColumn (Q1):** `select` returns exactly the columns you list. `withColumn` keeps every column and adds or replaces one. Both return a new DataFrame.
- **F.col (Q2):** `F.col("name")` refers to a column by name, so you can build expressions with it.
- **cast (Q3):** changes a column's data type, such as string → int. Values that can't be converted become null (ANSI off) or raise an error (ANSI on).
- **Type before arithmetic (Q4):** CSV gives text, and text compares character by character (`"10" < "5"`). Text maths may also quietly fall back to floating-point `double`, which loses exactness for money.
- **filter vs where (Q5):** the same function under two names.
- **& and | (Q6):** combine conditions with `&` (and), `|` (or) and `~` (not). Python's `and` / `or` don't work on columns.
- **Parentheses (Q7):** in Python, `&` and `|` bind tighter than `>` and `==`, so each comparison needs its own brackets: `(a > 0) & (b < 1)`.
- **when / otherwise (Q8):** SQL's `CASE WHEN`. Conditions are tested in order, the first true one wins, and `otherwise` is the default (null rows land there too, see Task 3).
- **isNull vs == None (Q9):** comparing with null gives null, never true, so `== None` matches nothing. Use `isNull()` or `eqNullSafe`.
- **Gross and net (Q10):** gross = quantity × unit_price; net = round(gross × (1 − discount_pct), 2), half-up.
- **Decimal precision (Q24):** multiplying adds precisions and scales, so intermediate results grow, as in Task 6. Cast the final value back to the storage type.
- **Negative quantity (Q27):** in this dataset, returns are recorded separately. A negative quantity is invalid input, and letting it reduce sales would hide a data error and double-count returns.